# Day 2.1 Access API Lab

In this lab, the local API provides fixed challenge endpoints. Each challenge endpoint looks like a realistic API you might inherit from another team: one may be rate-limited, one may be flaky, one may return a changed schema, and one may contain duplicate data.

Your job is to call the endpoint, observe the failure signal or data problem, and then write a client that handles it. The instructor/admin controls exist behind the scenes, but the student workflow is: **open the docs, call the challenge URL, inspect the response, fix the client**.


## 0. Start the Local API

Open a terminal in this folder and run:

```bash
pip install -r requirements.txt
python -m uvicorn mock_serving_api:app --reload --port 8011
```

Then open the API documentation:

```text
http://127.0.0.1:8011/docs
```

In [2]:
from pathlib import Path
import json
import time

import pandas as pd
import requests

BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = "group_03"  # change this for your group
OUTPUT_DIR = Path("api_lab_output")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMEOUT_SECONDS = 5

## 1. Read the API Status

Before collecting data, inspect the provider state. In a real project, this information usually comes from API documentation, status pages, or communication with the API provider.

In [3]:
response = requests.get(f"{BASE_URL}/api/v1/status", timeout=TIMEOUT_SECONDS)
response.raise_for_status()
response.json()

{'scenario': 'normal',
 'schema_version': 'v1',
 'request_count': 0,
 'last_request_at': None,
 'available_scenarios': ['normal',
  'flaky',
  'rate_limit',
  'timeout',
  'http_error',
  'duplicates'],
 'available_schema_versions': ['v1', 'v2'],
 'upstream_sources': {'taxi': 'https://api.data.gov.sg/v1/transport/taxi-availability',
  'weather': 'https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast',
  'rainfall': 'https://api-open.data.gov.sg/v2/real-time/api/rainfall'},
 'upstream_cache': {},
 'upstream_cache_ttl_seconds': 60,
 'rate_limit': {'max_requests': 3, 'window_seconds': 5}}

## 2. Day 1-Style Raw Taxi Endpoint

This endpoint looks like the Day 1 taxi API: a nested `FeatureCollection` with taxi coordinates.

Endpoint:

```text
GET /api/v1/transport/taxi-availability
```

In [4]:
payload = requests.get(
    f"{BASE_URL}/api/v1/transport/taxi-availability",
    params={"client_id": CLIENT_ID},
    timeout=TIMEOUT_SECONDS,
).json()

payload.keys(), len(payload["features"][0]["geometry"]["coordinates"])

(dict_keys(['type', 'crs', 'features']), 2703)

## 3. Task 1: Pagination

Most data APIs should not return everything in one response. This challenge endpoint returns one page at a time.

Collect all pages from:

```text
GET /debug-lab/06-pagination/taxi/records?page=1&page_size=10
```

Stop when `next_page` is `None`.


In [5]:
def normalize_record(record):
    record = dict(record)
    if "available_taxi_count" not in record and "taxi_count" in record:
        record["available_taxi_count"] = record.pop("taxi_count")
    if "rainfall_mm" not in record and "rainfall_value_mm" in record:
        record["rainfall_mm"] = record.pop("rainfall_value_mm")
    return record


def collect_all_pages(url, page_size=10):
    page = 1
    records = []

    while True:
        response = requests.get(
            url,
            params={
                "page": page,
                "page_size": page_size,
                "client_id": CLIENT_ID,
            },
            timeout=TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()

        records.extend(normalize_record(item) for item in payload["data"])
        if payload["next_page"] is None:
            break
        page = payload["next_page"]

    return records


records = collect_all_pages(
    f"{BASE_URL}/debug-lab/06-pagination/taxi/records",
    page_size=15,
)
df = pd.DataFrame(records)
df.head(), df.shape


(                          record_id              api_timestamp  taxi_index  \
 0  taxi:2026-06-23T09:35:46+08:00:1  2026-06-23T09:35:46+08:00           1   
 1  taxi:2026-06-23T09:35:46+08:00:2  2026-06-23T09:35:46+08:00           2   
 2  taxi:2026-06-23T09:35:46+08:00:3  2026-06-23T09:35:46+08:00           3   
 3  taxi:2026-06-23T09:35:46+08:00:4  2026-06-23T09:35:46+08:00           4   
 4  taxi:2026-06-23T09:35:46+08:00:5  2026-06-23T09:35:46+08:00           5   
 
    longitude  latitude  available_taxi_count                         source  
 0  103.61394   1.25283                  2703  data.gov.sg taxi availability  
 1  103.61394   1.25284                  2703  data.gov.sg taxi availability  
 2  103.62244   1.28217                  2703  data.gov.sg taxi availability  
 3  103.62292   1.28347                  2703  data.gov.sg taxi availability  
 4  103.62297   1.29881                  2703  data.gov.sg taxi availability  ,
 (2703, 7))

In [6]:
df.to_csv(OUTPUT_DIR / "taxi_paginated.csv", index=False)
df["record_id"].nunique(), len(df)

(2703, 2703)

## 4. Task 2: Retry With Exponential Backoff

This challenge endpoint is flaky. The first two requests for each `client_id` return `503`, then the same request succeeds.

Endpoint:

```text
GET /debug-lab/04-flaky/rainfall/records?page=1&page_size=5
```

A robust client should retry temporary failures instead of failing immediately.


In [7]:
retry_client_id = f"{CLIENT_ID}_retry_demo_{int(time.time())}"
retry_client_id


'group_03_retry_demo_1782178851'

In [8]:
# These status codes usually mean the problem may be temporary.
# Instead of failing immediately, the client can wait and retry.
TEMPORARY_STATUS_CODES = {429, 500, 502, 503, 504}


def get_with_retry(url, params=None, max_attempts=5, base_sleep=1):
    # Try the same request multiple times before giving up.
    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}: sending request to the API")
        try:
            response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
            print(f"Received status code: {response.status_code}")

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", base_sleep))
                print(f"Request was rate limited. Waiting {retry_after}s before retrying")
                time.sleep(retry_after)
                continue

            if response.status_code in TEMPORARY_STATUS_CODES:
                wait_seconds = base_sleep * (2 ** (attempt - 1))
                print(f"Temporary API error: HTTP {response.status_code}")
                print(f"Retrying after {wait_seconds}s")
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            print("Request successful")
            return response
        except requests.Timeout:
            wait_seconds = base_sleep * (2 ** (attempt - 1))
            print("Request timed out")
            print(f"Retrying after {wait_seconds}s")
            time.sleep(wait_seconds)

    print("No retry attempts left")
    raise RuntimeError(f"Request failed after {max_attempts} attempts: {url}")


response = get_with_retry(
    f"{BASE_URL}/debug-lab/04-flaky/rainfall/records",
    params={"page": 1, "page_size": 5, "client_id": retry_client_id},
)
response.json()["data"][:2]

Attempt 1/5: sending request to the API
Received status code: 503
Temporary API error: HTTP 503
Retrying after 1s
Attempt 2/5: sending request to the API
Received status code: 503
Temporary API error: HTTP 503
Retrying after 2s
Attempt 3/5: sending request to the API
Received status code: 200
Request successful


[{'record_id': 'rainfall:2026-06-23T09:35:00+08:00:S218',
  'reading_timestamp': '2026-06-23T09:35:00+08:00',
  'station_id': 'S218',
  'station_name': 'Bukit Batok Street 34',
  'longitude': 103.75065,
  'latitude': 1.36491,
  'rainfall_mm': 0,
  'source': 'data.gov.sg rainfall'},
 {'record_id': 'rainfall:2026-06-23T09:35:00+08:00:S219',
  'reading_timestamp': '2026-06-23T09:35:00+08:00',
  'station_id': 'S219',
  'station_name': 'Yio Chu Kang Road',
  'longitude': 103.87639,
  'latitude': 1.37998,
  'rainfall_mm': 0,
  'source': 'data.gov.sg rainfall'}]

## 5. Task 3: Rate Limiting

This challenge endpoint allows only 3 requests per 5 seconds for each client. Extra requests return `429` with a `Retry-After` header.

Endpoint:

```text
GET /debug-lab/03-rate-limit/weather/records?page=1&page_size=3
```


In [ ]:
rate_limit_client_id = f"{CLIENT_ID}_rate_limit_demo_{int(time.time())}"

for i in range(6):
    response = get_with_retry(
        f"{BASE_URL}/debug-lab/03-rate-limit/weather/records",
        params={"page": 1, "page_size": 3, "client_id": rate_limit_client_id},
        max_attempts=3,
    )
    print(i, response.status_code, len(response.json()["data"]))


Attempt 1/1: sending request to the API
Received status code: 200
Request successful
0 200 3
Attempt 1/1: sending request to the API
Received status code: 200
Request successful
1 200 3
Attempt 1/1: sending request to the API
Received status code: 200
Request successful
2 200 3
Attempt 1/1: sending request to the API
Received status code: 429
Request was rate limited. Waiting 5s before retrying
No retry attempts left


RuntimeError: Request failed after 1 attempts: http://127.0.0.1:8011/debug-lab/03-rate-limit/weather/records

## 6. Task 4: Schema Change

This challenge endpoint returns schema `v2`. In `v1`, taxi records contain `available_taxi_count`. In `v2`, the same value is renamed to `taxi_count`.

Endpoint:

```text
GET /debug-lab/02-schema-change/taxi/records?page=1&page_size=5
```

Your client should normalize the field name so downstream code still receives `available_taxi_count`.


In [ ]:
response = get_with_retry(
    f"{BASE_URL}/debug-lab/02-schema-change/taxi/records",
    params={"page": 1, "page_size": 5, "client_id": CLIENT_ID},
)
payload = response.json()
raw_items = payload["data"]
normalized_items = [normalize_record(item) for item in raw_items]

payload["schema_version"], pd.DataFrame(raw_items).head(), pd.DataFrame(normalized_items).head()


Attempt 1/5: sending request to the API
Received status code: 200
Request successful


('v2',
                           record_id              api_timestamp  taxi_index  \
 0  taxi:2026-06-22T22:36:49+08:00:1  2026-06-22T22:36:49+08:00           1   
 1  taxi:2026-06-22T22:36:49+08:00:2  2026-06-22T22:36:49+08:00           2   
 2  taxi:2026-06-22T22:36:49+08:00:3  2026-06-22T22:36:49+08:00           3   
 3  taxi:2026-06-22T22:36:49+08:00:4  2026-06-22T22:36:49+08:00           4   
 4  taxi:2026-06-22T22:36:49+08:00:5  2026-06-22T22:36:49+08:00           5   
 
     longitude  latitude                         source  taxi_count  \
 0  103.637980  1.323460  data.gov.sg taxi availability        3323   
 1  103.638010  1.323450  data.gov.sg taxi availability        3323   
 2  103.643748  1.325880  data.gov.sg taxi availability        3323   
 3  103.646368  1.326671  data.gov.sg taxi availability        3323   
 4  103.646368  1.326671  data.gov.sg taxi availability        3323   
 
                                          schema_note  
 0  schema v2 renamed available_t

## 7. Task 5: Deduplication

This challenge endpoint returns duplicate `record_id` values. Collect all pages, then keep one row per `record_id`.

Endpoint:

```text
GET /debug-lab/05-duplicates/taxi/records?page=1&page_size=20
```


In [11]:
records = collect_all_pages(
    f"{BASE_URL}/debug-lab/05-duplicates/taxi/records",
    page_size=20,
)
raw_df = pd.DataFrame(records)
clean_df = raw_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)

raw_df.to_csv(OUTPUT_DIR / "duplicates_raw.csv", index=False)
clean_df.to_csv(OUTPUT_DIR / "duplicates_clean.csv", index=False)

len(raw_df), len(clean_df), clean_df["record_id"].nunique()


(3013, 3000, 3000)

## 8. Task 6: Checkpoint / Resume

Long collection jobs may stop halfway. This challenge endpoint behaves like a normal paginated API, so the failure is in our client process rather than the provider response.

Endpoint:

```text
GET /debug-lab/08-checkpoint/taxi/records?page=1&page_size=10
```

This example writes:

- `checkpoint.json`: last completed page
- `checkpoint_records.jsonl`: records already collected


In [12]:
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"
CHECKPOINT_RECORDS_FILE = OUTPUT_DIR / "checkpoint_records.jsonl"


def read_checkpoint():
    if not CHECKPOINT_FILE.exists():
        return {"last_completed_page": 0}
    return json.loads(CHECKPOINT_FILE.read_text(encoding="utf-8"))


def write_checkpoint(last_completed_page):
    CHECKPOINT_FILE.write_text(
        json.dumps({"last_completed_page": last_completed_page}, indent=2),
        encoding="utf-8",
    )


def append_records_jsonl(records):
    with CHECKPOINT_RECORDS_FILE.open("a", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record) + "\n")


def collect_with_checkpoint(page_size=10, simulate_crash_after_page=None):
    checkpoint = read_checkpoint()
    page = checkpoint["last_completed_page"] + 1

    while True:
        response = get_with_retry(
            f"{BASE_URL}/debug-lab/08-checkpoint/taxi/records",
            params={"page": page, "page_size": page_size, "client_id": CLIENT_ID},
        )
        payload = response.json()
        records = [normalize_record(item) for item in payload["data"]]

        append_records_jsonl(records)
        write_checkpoint(page)
        print(f"completed page {page}")

        if simulate_crash_after_page == page:
            raise RuntimeError("simulated crash after writing checkpoint")

        if payload["next_page"] is None:
            break
        page = payload["next_page"]


# Uncomment to start from scratch.
# CHECKPOINT_FILE.unlink(missing_ok=True)
# CHECKPOINT_RECORDS_FILE.unlink(missing_ok=True)

In [13]:
# First run: simulate a crash.
try:
    collect_with_checkpoint(page_size=10, simulate_crash_after_page=3)
except RuntimeError as exc:
    print(exc)

read_checkpoint()

Attempt 1/5: sending request to the API
Received status code: 200
Request successful
completed page 1
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
completed page 2
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
completed page 3
simulated crash after writing checkpoint


{'last_completed_page': 3}

In [ ]:
# Second run: resume from the last completed page.
collect_with_checkpoint(page_size=10, simulate_crash_after_page=None)

checkpoint_df = pd.read_json(CHECKPOINT_RECORDS_FILE, lines=True)
checkpoint_clean_df = checkpoint_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)
checkpoint_clean_df.shape, checkpoint_clean_df["record_id"].nunique()

## 9. Final Validation

Use the provider's validation endpoint to check the expected row counts.

In [14]:
expected = requests.get(f"{BASE_URL}/api/v1/validation/expected", timeout=TIMEOUT_SECONDS).json()
expected

{'schema_v1_taxi_count_field': 'available_taxi_count',
 'schema_v2_taxi_count_field': 'taxi_count',
 'schema_v1_rainfall_field': 'rainfall_mm',
 'schema_v2_rainfall_field': 'rainfall_value_mm',
 'taxi_base_total_records': 3272,
 'taxi_duplicate_total_records': 3285,
 'taxi_unique_record_ids': 3272,
 'weather_base_total_records': 47,
 'weather_duplicate_total_records': 60,
 'weather_unique_record_ids': 47,
 'rainfall_base_total_records': 77,
 'rainfall_duplicate_total_records': 90,
 'rainfall_unique_record_ids': 77}

## 10. Provider-Side Code Reading

After your client works, open `mock_serving_api.py` and find these pieces:

1. Where does FastAPI validate `page` and `page_size`?
2. Where does the provider return `429` and `Retry-After`?
3. Where does the schema change from `available_taxi_count` to `taxi_count`?
4. Why does the provider include pagination metadata?
